In [133]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

In [134]:
df = pd.read_csv('Travel.csv')
df.head()

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,3.0,Deluxe,3.0,Single,1.0,1,2,1,0.0,Manager,20993.0
1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,3,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,4.0,Basic,3.0,Single,7.0,1,3,0,0.0,Executive,17090.0
3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,2,3.0,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,200004,0,NaN,Self Enquiry,1,8.0,Small Business,Male,2,3.0,Basic,4.0,Divorced,1.0,0,5,1,0.0,Executive,18468.0


In [135]:
print(f"Rows : {df.shape[0]}, Columns : {df.shape[1]}")

Rows : 4888, Columns : 20


In [136]:
df.dtypes

CustomerID                    int64
ProdTaken                     int64
Age                         float64
TypeofContact                object
CityTier                      int64
DurationOfPitch             float64
Occupation                   object
Gender                       object
NumberOfPersonVisiting        int64
NumberOfFollowups           float64
ProductPitched               object
PreferredPropertyStar       float64
MaritalStatus                object
NumberOfTrips               float64
Passport                      int64
PitchSatisfactionScore        int64
OwnCar                        int64
NumberOfChildrenVisiting    float64
Designation                  object
MonthlyIncome               float64
dtype: object

In [137]:
df.sample()

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
1005,201005,0,29.0,Self Enquiry,2,23.0,Salaried,Male,2,4.0,Standard,5.0,Unmarried,2.0,0,3,0,0.0,Senior Manager,22988.0


In [138]:
df = df.drop(columns = ['CustomerID'])

In [139]:
df.sample()

,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
3715,1,22.0,Self Enquiry,3,14.0,Small Business,Male,3,4.0,Basic,4.0,Unmarried,3.0,0,5,1,2.0,Executive,21357.0


In [140]:
df.isnull().sum()

ProdTaken                     0
Age                         226
TypeofContact                25
CityTier                      0
DurationOfPitch             251
Occupation                    0
Gender                        0
NumberOfPersonVisiting        0
NumberOfFollowups            45
ProductPitched                0
PreferredPropertyStar        26
MaritalStatus                 0
NumberOfTrips               140
Passport                      0
PitchSatisfactionScore        0
OwnCar                        0
NumberOfChildrenVisiting     66
Designation                   0
MonthlyIncome               233
dtype: int64

In [141]:
for col in df.columns:
    if df[col].dtype in ['float64','int64']:
        median_value = df[col].median()
        df[col] = df[col].fillna(median_value)
    else:
        most_common = df[col].mode()[0]
        df[col] = df[col].fillna(most_common)

In [142]:
df.isnull().sum()

ProdTaken                   0
Age                         0
TypeofContact               0
CityTier                    0
DurationOfPitch             0
Occupation                  0
Gender                      0
NumberOfPersonVisiting      0
NumberOfFollowups           0
ProductPitched              0
PreferredPropertyStar       0
MaritalStatus               0
NumberOfTrips               0
Passport                    0
PitchSatisfactionScore      0
OwnCar                      0
NumberOfChildrenVisiting    0
Designation                 0
MonthlyIncome               0
dtype: int64

In [143]:
print('Dataset shape after cleaning:', df.shape)

Dataset shape after cleaning: (4888, 19)


In [144]:
df.describe()

,ProdTaken,Age,CityTier,DurationOfPitch,NumberOfPersonVisiting,NumberOfFollowups,PreferredPropertyStar,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,MonthlyIncome
count,4888.000000,4888.000000,4888.000000,4888.000000,4888.000000,4888.000000,4888.000000,4888.000000,4888.000000,4888.000000,4888.000000,4888.000000,4888.000000
mean,0.188216,37.547259,1.654255,15.362930,2.905074,3.711129,3.577946,3.229746,0.290917,3.078151,0.620295,1.184738,23559.179419
std,0.390925,9.104795,0.916583,8.316166,0.724891,0.998271,0.797005,1.822769,0.454232,1.365792,0.485363,0.852323,5257.862921
min,0.000000,18.000000,1.000000,5.000000,1.000000,1.000000,3.000000,1.000000,0.000000,1.000000,0.000000,0.000000,1000.000000
25%,0.000000,31.000000,1.000000,9.000000,2.000000,3.000000,3.000000,2.000000,0.000000,2.000000,0.000000,1.000000,20485.000000
50%,0.000000,36.000000,1.000000,13.000000,3.000000,4.000000,3.000000,3.000000,0.000000,3.000000,1.000000,1.000000,22347.000000
75%,0.000000,43.000000,3.000000,19.000000,3.000000,4.000000,4.000000,4.000000,1.000000,4.000000,1.000000,2.000000,25424.750000
max,1.000000,61.000000,3.000000,127.000000,5.000000,6.000000,5.000000,22.000000,1.000000,5.000000,1.000000,3.000000,98678.000000


In [145]:
plt.figure(figsize=(12, 8))
df['ProdTaken'].value_counts().plot(kind='bar', color=['skyblue', 'salmon'])
plt.title('Target : Did customer buy Travel Package?')
plt.xlabel("0 = No, 1 = Yes")
plt.ylabel("Count")
plt.xticks()
plt.tight_layout()
plt.savefig('eda_1_target_distribution.png')
plt.close()
print('Saved: eda_1_target_distribution.png')

Saved: eda_1_target_distribution.png


In [146]:
plt.figure(figsize=(12, 8))
sns.histplot(data=df, x='Age', hue='ProdTaken', bins=20, kde=True)
plt.title('Age Distribution by Target')
plt.savefig('eda_2_age_distribution.png')
plt.close()
print('Saved: eda_2_age_distribution.png')

Saved: eda_2_age_distribution.png


In [147]:
df.sample()

,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
4256,0,32.0,Self Enquiry,1,14.0,Small Business,Fe Male,3,4.0,Standard,3.0,Unmarried,3.0,1,4,1,2.0,Senior Manager,25821.0


In [148]:
plt.figure(figsize=(12, 8))
sns.boxplot(data=df, x='ProdTaken', y='MonthlyIncome')
plt.title('Monthly Income vs Product Taken')
plt.xlabel("0 = No, 1 = Yes")
plt.savefig('eda_3_monthly_income_boxplot.png')
plt.close()
print('Saved: eda_3_monthly_income_boxplot.png')

Saved: eda_3_monthly_income_boxplot.png


In [149]:
plt.figure(figsize=(12, 8))
numeric_df = df.select_dtypes(include=['float64', 'int64'])
correlation_matrix = numeric_df.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix')
plt.savefig('eda_4_correlation_matrix.png')
plt.close()
print('Saved: eda_4_correlation_matrix.png')

Saved: eda_4_correlation_matrix.png


In [150]:
le = LabelEncoder()
text_cols = [col for col in df.columns if df[col].dtype == 'object']
text_cols

['TypeofContact',
 'Occupation',
 'Gender',
 'ProductPitched',
 'MaritalStatus',
 'Designation']

In [151]:
for col in text_cols:
    df[col] = le.fit_transform(df[col])
    print(f"Encoded {col} with classes: {le.classes_}")

Encoded TypeofContact with classes: ['Company Invited' 'Self Enquiry']
Encoded Occupation with classes: ['Free Lancer' 'Large Business' 'Salaried' 'Small Business']
Encoded Gender with classes: ['Fe Male' 'Female' 'Male']
Encoded ProductPitched with classes: ['Basic' 'Deluxe' 'King' 'Standard' 'Super Deluxe']
Encoded MaritalStatus with classes: ['Divorced' 'Married' 'Single' 'Unmarried']
Encoded Designation with classes: ['AVP' 'Executive' 'Manager' 'Senior Manager' 'VP']


In [152]:
df.head()

,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,1,41.0,1,3,6.0,2,1,3,3.0,1,3.0,2,1.0,1,2,1,0.0,2,20993.0
1,0,49.0,0,1,14.0,2,2,3,4.0,1,4.0,0,2.0,0,3,1,2.0,2,20130.0
2,1,37.0,1,1,8.0,0,2,3,4.0,0,3.0,2,7.0,1,3,0,0.0,1,17090.0
3,0,33.0,0,1,9.0,2,1,2,3.0,0,3.0,0,2.0,1,5,1,1.0,1,17909.0
4,0,36.0,1,1,8.0,3,2,2,3.0,0,4.0,0,1.0,0,5,1,0.0,1,18468.0


In [153]:
X = df.drop(columns=['ProdTaken'])
y = df['ProdTaken']

In [154]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)

In [155]:
print(f"Training rows: {X_train.shape[0]}, Testing rows: {X_test.shape[0]}")

Training rows: 3910, Testing rows: 978


In [156]:
scaler = StandardScaler()
X_trained_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [157]:
X_train_scaled = pd.DataFrame(X_trained_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

In [158]:
X_train_scaled

,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
619,-0.173791,0.642318,1.477328,-1.116889,1.046192,0.788785,-1.256729,-2.708464,-0.147914,1.775141,-0.297105,-0.673593,-0.643510,0.670552,-1.279276,-1.379190,0.285082,-0.229477
1227,-0.173791,0.642318,-0.711099,-0.879814,-0.521092,0.788785,0.128223,-0.708068,-0.928408,-0.729659,-0.297105,-0.673593,-0.643510,1.401078,-1.279276,0.964414,-0.755693,-0.959931
3238,-0.283409,-1.556862,-0.711099,-0.642739,-0.521092,0.788785,0.128223,0.292129,-0.147914,-0.729659,1.856147,-0.673593,-0.643510,-0.790500,0.781692,-0.207388,0.285082,0.541936
3961,0.045445,0.642318,-0.711099,1.253859,-0.521092,0.788785,1.513175,0.292129,-0.928408,0.522741,-0.297105,1.518913,-0.643510,0.670552,-1.279276,0.964414,-0.755693,-0.351596
490,0.045445,0.642318,-0.711099,-1.116889,-0.521092,-1.017980,-1.256729,-0.708068,-0.928408,1.775141,0.779521,0.422660,-0.643510,-0.790500,0.781692,-0.207388,-0.755693,-1.121876
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1403,0.155064,0.642318,1.477328,0.661172,1.046192,0.788785,0.128223,-0.708068,2.193566,1.775141,1.856147,-1.221719,-0.643510,-1.521026,0.781692,0.964414,-1.796467,0.901878
492,1.141628,0.642318,-0.711099,-1.116889,-0.521092,-1.017980,-1.256729,0.292129,2.193566,1.775141,-1.373730,-1.221719,-0.643510,0.670552,-1.279276,-0.207388,-1.796467,1.290887
2969,-1.379592,0.642318,1.477328,-0.642739,-0.521092,-1.017980,1.513175,0.292129,-0.147914,-0.729659,1.856147,-0.673593,-0.643510,-0.790500,0.781692,-0.207388,0.285082,-0.058094
2849,-1.269974,0.642318,-0.711099,-0.642739,1.046192,0.788785,1.513175,0.292129,-0.928408,1.775141,-1.373730,2.067039,-0.643510,1.401078,0.781692,0.964414,-0.755693,-0.161150


In [159]:
# (Name, Model, UsedScaledData)
models = [
    ('KNN Classifier', KNeighborsClassifier(n_neighbors=5), True),
    ('LogisticRegression', LogisticRegression(max_iter=1000, random_state=42), True),
    ('Naive Bayes', GaussianNB(), False),
    ('Decision Tree', DecisionTreeClassifier(random_state=42), False),
    ('SVM Classifier', SVC(random_state=42), True),
    ('Random Forest', RandomForestClassifier(random_state=42), False),
    ('AdaBoost', AdaBoostClassifier(random_state=42), False)
]

In [160]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['ProdTaken']), df['ProdTaken'], test_size=0.2, random_state=42)

In [161]:
results = {}
for name, model, use_scaled in models:
    if use_scaled:
        X_tr, X_te = X_trained_scaled, X_test_scaled
    else:
        X_tr, X_te = X_train, X_test
        
    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    
    print(f"\n{'--' * 50}")
    print(f"Model: {name}")
    print(f"Accuracy: {acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Not Taken', 'Taken']))

c:\Users\N I T R O V15\.conda\envs\ga\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but KNeighborsClassifier was fitted without feature names
  warnings.warn(
c:\Users\N I T R O V15\.conda\envs\ga\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
c:\Users\N I T R O V15\.conda\envs\ga\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\N I T R O V15\.conda\envs\ga\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf


----------------------------------------------------------------------------------------------------
Model: KNN Classifier
Accuracy: 0.7771

Classification Report:
              precision    recall  f1-score   support

   Not Taken       0.80      0.96      0.87       787
       Taken       0.14      0.03      0.04       191

    accuracy                           0.78       978
   macro avg       0.47      0.49      0.46       978
weighted avg       0.67      0.78      0.71       978


----------------------------------------------------------------------------------------------------
Model: LogisticRegression
Accuracy: 0.8047

Classification Report:
              precision    recall  f1-score   support

   Not Taken       0.80      1.00      0.89       787
       Taken       0.00      0.00      0.00       191

    accuracy                           0.80       978
   macro avg       0.40      0.50      0.45       978
weighted avg       0.65      0.80      0.72       978


-----------

c:\Users\N I T R O V15\.conda\envs\ga\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but SVC was fitted without feature names
  warnings.warn(



----------------------------------------------------------------------------------------------------
Model: SVM Classifier
Accuracy: 0.8047

Classification Report:
              precision    recall  f1-score   support

   Not Taken       0.80      1.00      0.89       787
       Taken       0.00      0.00      0.00       191

    accuracy                           0.80       978
   macro avg       0.40      0.50      0.45       978
weighted avg       0.65      0.80      0.72       978



c:\Users\N I T R O V15\.conda\envs\ga\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\N I T R O V15\.conda\envs\ga\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\N I T R O V15\.conda\envs\ga\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



----------------------------------------------------------------------------------------------------
Model: Random Forest
Accuracy: 0.9294

Classification Report:
              precision    recall  f1-score   support

   Not Taken       0.93      0.99      0.96       787
       Taken       0.96      0.67      0.79       191

    accuracy                           0.93       978
   macro avg       0.94      0.83      0.87       978
weighted avg       0.93      0.93      0.92       978


----------------------------------------------------------------------------------------------------
Model: AdaBoost
Accuracy: 0.8395

Classification Report:
              precision    recall  f1-score   support

   Not Taken       0.84      0.98      0.91       787
       Taken       0.79      0.24      0.37       191

    accuracy                           0.84       978
   macro avg       0.82      0.61      0.64       978
weighted avg       0.83      0.84      0.80       978



In [162]:
results

{'KNN Classifier': 0.7770961145194274,
 'LogisticRegression': 0.8047034764826176,
 'Naive Bayes': 0.8139059304703476,
 'Decision Tree': 0.9202453987730062,
 'SVM Classifier': 0.8047034764826176,
 'Random Forest': 0.9294478527607362,
 'AdaBoost': 0.8394683026584867}

In [163]:
print('Model Comparison Plot: ')
print('=' * 50)
model_names = list(results.keys())
accuracies = list(results.values())

plt.figure(figsize=(12, 8))
bars = plt.barh(model_names, accuracies, color='skyblue')
plt.xlabel('Accuracy')

for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, f'{acc:.4f}', va='center')

plt.tight_layout()
plt.savefig('model_comparison.png')
plt.close()
print('Saved: model_comparison.png')

Model Comparison Plot: 
Saved: model_comparison.png


In [164]:
results

{'KNN Classifier': 0.7770961145194274,
 'LogisticRegression': 0.8047034764826176,
 'Naive Bayes': 0.8139059304703476,
 'Decision Tree': 0.9202453987730062,
 'SVM Classifier': 0.8047034764826176,
 'Random Forest': 0.9294478527607362,
 'AdaBoost': 0.8394683026584867}

In [165]:
best_model_name = max(results, key=results.get)
print(f"Best Model: {best_model_name} with Accuracy: {results[best_model_name]:.4f}")

Best Model: Random Forest with Accuracy: 0.9294


In [166]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10]
}

In [167]:
rf_model = RandomForestClassifier(random_state=42)

In [168]:
grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

In [169]:
grid_search.fit(X_train, y_train)

Fitting 3 folds for each of 36 candidates, totalling 108 fits


GridSearchCV(cv=3, estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [None, 10, 20, 30],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [50, 100, 200]},
             scoring='accuracy', verbose=1)

In [170]:
print("Best Parameters from Grid Search:", grid_search.best_params_)
print('Best cross-validation accuracy:', grid_search.best_score_)

Best Parameters from Grid Search: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 50}
Best cross-validation accuracy: 0.9127881701343603


In [171]:
best_rf_model = grid_search.best_estimator_
y_pred_best = best_rf_model.predict(X_test)
tuned_acc = accuracy_score(y_test, y_pred_best)
print(f"Tuned Random Forest Accuracy on Test Set: {tuned_acc:.4f}")
print("\nClassification Report for Tuned Random Forest:")
print(classification_report(y_test, y_pred_best, target_names=['Not Taken', 'Taken']))

Tuned Random Forest Accuracy on Test Set: 0.9274

Classification Report for Tuned Random Forest:
              precision    recall  f1-score   support

   Not Taken       0.92      0.99      0.96       787
       Taken       0.95      0.66      0.78       191

    accuracy                           0.93       978
   macro avg       0.94      0.83      0.87       978
weighted avg       0.93      0.93      0.92       978



In [172]:
cm = confusion_matrix(y_test, y_pred_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Not Taken', 'Taken'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix for Tuned Random Forest')
plt.tight_layout()
plt.savefig('tuned_rf_confusion_matrix.png')
plt.close()
print('Saved: tuned_rf_confusion_matrix.png')

Saved: tuned_rf_confusion_matrix.png


In [173]:
joblib.dump(best_rf_model, 'best_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
print('Saved: best_model.pkl and scaler.pkl')

Saved: best_model.pkl and scaler.pkl


In [175]:
loaded_scaler = joblib.load('scaler.pkl')
loaded_model = joblib.load('best_model.pkl')

In [176]:
new_customer = pd.DataFrame({
    '''
    feature1: [value1],
    feature2: [value2],
    '''
})